
![](https://raw.githubusercontent.com/wateraccounting/WaPORMOOC/main/images/banner_notebooks_WaPOR4Global.png)

[![](https://raw.githubusercontent.com//wateraccounting/WaPORMOOC/main/images/colab-badge.png)](https://colab.research.google.com/github/wateraccounting/WaPOR4Global/blob/main/5_Creating_Dashboard/Notebook_5_Dashboard_WaPOR4Global_using_Bokeh.ipynb?target="_blank")

<div style="text-align:center; margin-top:15px;">
<h2 style="margin-bottom:5px;">
  Compare climate data for Iraq in a dashboard &nbsp;  
  <a href="https://bokeh.org/" target="_blank">
    <img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcQ3drsYnILnkhhBCjg1awMCqD2FzekFOJeHcQ&s"
     height="30" width="auto" style="vertical-align:middle; margin-left:8px;">
  </a>
</h2>
<p style="font-size:14px; margin-top:8px;">
  📚 Explore Bokeh tutorials and documentation at
  <a href="https://bokeh.org/" target="_blank" style="color:#F5A623;">bokeh.org</a>  &nbsp;
</p>
</div>
<p style="margin-top:5px;">
<b>Notebook 5:</b> Creating dashboard within Colab using Bokeh &nbsp; | &nbsp;
<b>Estimated time:</b> 2 hours &nbsp; | &nbsp;
<b>Instructor:</b> Dr. Marloes Mul & Laura Agudelo Mayorga
</p>
</div>

### **What is Bokeh?**

Bokeh is an open-source library for creating **interactive visualization for modern web
browsers**.

With Bokeh, you can use Python to build beautiful data visualizations, ranging from
simple plots to complex dashboards with streaming datasets.
In a nutshell: Bokeh lets you create JavaScript-powered visualizations **without having
to write any JavaScript yourself**.

Bokeh can generate
[stand-alone HTML objects](https://docs.bokeh.org/en/latest/docs/user_guide/output/embed.html)
to use in any kind of website, or you can run Bokeh as a
[server](http://docs.bokeh.org/en/latest/docs/user_guide/server.html).
But Bokeh also works directly in Jupyter/Colab Notebooks.


### What's in this tutorial?

This notebook is an **interactive** way to explore data visualization in Python — comparing static and interactive plots, and understanding when to use each.

We will work through two approaches:

**1. Static visualizations** using:
```python
import matplotlib.pyplot as plt
import seaborn as sns
```
These are ideal for reports, publications, and quick exploratory analysis — clean, simple, and easy to export.

**2. Interactive visualizations** using Bokeh, which allow you to zoom, pan, hover, and filter data directly in the notebook — perfect for dashboards and presentations where the audience can explore the data themselves.
```python
from bokeh.io import output_notebook, show

output_notebook()
```
---

### What we'll do in this notebook:

1. 📂 Load the files and explore the dataset
2. 📊 Create **static plots** with Matplotlib & Seaborn
3. ✨ Recreate similar plots as **interactive visualizations** with Bokeh
4. 🔍 Compare both approaches and discuss their use cases

👇 Run the code cell below to get started!


### ⚙️ Environment Setup

In [ ]:
# =============================================================================
# activate Bokeh output in Colab notebook
# =============================================================================

from bokeh.io import output_notebook, show
from bokeh.resources import INLINE # Ensure INLINE is imported

output_notebook(INLINE) #ensures the plots always render correctly, regardless of internet access.

In [ ]:
# =============================================================================
# INSTALL & IMPORT PACKAGES
# =============================================================================

# Uncomment to install if needed:
!pip install numpy pandas matplotlib seaborn scipy --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import seaborn as sns
import geopandas as gpd
from scipy import stats
import os
import warnings
from datetime import datetime
import json
from scipy.stats import pearsonr


print(f"\n✅ Setup complete — Pandas {pd.__version__}, NumPy {np.__version__}")
print(f"📅 Analysis date: {datetime.now().strftime('%Y-%m-%d')}")

In [ ]:
# =============================================================================
# INSTALL & IMPORT PACKAGES FROM BOKEH
# =============================================================================

from bokeh.plotting import figure,show
from bokeh.models import ColumnDataSource, HoverTool, Label, CDSView, IndexFilter, Range1d, CustomJS, Select
from bokeh.transform import factor_cmap
from bokeh.palettes import Category20
from bokeh.layouts import gridplot
from bokeh.layouts import column, row as bokeh_row # Renamed 'row' to 'bokeh_row' to avoid conflict with potential DataFrame variable named 'row'
from bokeh.models import (
    GeoJSONDataSource, LinearColorMapper, ColorBar,
    RadioButtonGroup,Div
)
from bokeh.palettes import Blues256, RdBu11, Greens256

### Data Files
Create folders for where to upload the input data and where to store the results.

In [3]:
# =============================================================================
# CREATE INPUT AND OUTPUT FOLDERS
# =============================================================================

data_folder = 'data'
os.makedirs(data_folder, exist_ok=True)

output_folder = 'output_data'
os.makedirs(output_folder, exist_ok=True)

### Upload the data files
Place the following data files in a new folder called `data/` from Notebook 1 folder output *irq_admin1*

| File | Description |
|------|-------------|
| `AgERA5-PCP-M_mm.csv` | Monthly rainfall (mm/month) for 18 governorates of Iraq|
| `AgERA5-RET-M_mm.csv` | Monthly reference evapotranspiration (mm/month) for 18 governorates of Iraq|
| `L1-PCP-M_mm_per_month.csv` | Monthly rainfall (mm/month) for 18 governorates of Iraq|
| `L1-RET-M_mm_per_month.csv` |Monthly reference evapotranspiration (mm/month) for 18 governorates of Iraq|
| `irq_admin1.geojsonv` |Shapefile from 18 governorates in Iraq|

**Expected CSV format:** First column = date (`DD/MM/YYYY`), remaining columns = governorate names.

In [ ]:
# =============================================================================
# UPLOAD DATA
# =============================================================================
from google.colab import files
uploaded = files.upload()

# Save each file to the destination folder
for filename, content in uploaded.items():
    dest_path = f'/content/data/{filename}'
    with open(dest_path, 'wb') as f:
        f.write(content)
    print(f" Saved: {dest_path}")

In [7]:
# =============================================================================
# DATA LOADING — WAPOR MONTHLY
# =============================================================================

def load_governorate_data(filepath, name='dataset'):
    """
    Load monthly governorate-level data from CSV.
    Expects: first column = date, remaining columns = governorates.
    """
    # Changed 'delimiter=';'' to 'delimiter=',' to handle comma-separated CSVs
    df = pd.read_csv(filepath, delimiter=',')
    date_col = df.columns[0]

    # Parse dates (try multiple formats)
    for fmt in ['%d/%m/%Y', '%m/%d/%Y', '%Y-%m-%d', '%Y/%m/%d']:
        try:
            df[date_col] = pd.to_datetime(df[date_col], format=fmt)
            break
        except (ValueError, TypeError):
            continue
    else:
        df[date_col] = pd.to_datetime(df[date_col], dayfirst=True)

    df.set_index(date_col, inplace=True)
    df.index.name = 'date'
    df = df.sort_index()

    # Ensure numeric
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    print(f"\U0001f4c4 {name}: {df.shape[0]} months × {df.shape[1]} governorates")
    print(f"   Period: {df.index.min().strftime('%b %Y')} — {df.index.max().strftime('%b %Y')}")
    return df

In [ ]:
# =============================================================================
# LOADING WAPOR AND AgERA5 CLIMATE DATA FOR ALL GOVERNORATES
# =============================================================================

# --- Load AgERA5 Rainfall ---
rainfall_AgERA = os.path.join(data_folder, '/content/data/irq_admin1_AgERA5-PCP-M_mm.csv') #change your paths in case of having different names
df_rain_AgERA = load_governorate_data(rainfall_AgERA, name='Rainfall_AgERA')

# --- Load AgERA5 Reference ET ---
ret_AgERA = os.path.join('/content/data/irq_admin1_AgERA5-RET-M_mm.csv')
df_ret_AgERA = load_governorate_data(ret_AgERA, name='RET_AgERA')

# --- Load WaPOR Rainfall ---
rainfall_WaPOR = os.path.join('/content/data/irq_admin1_L1-PCP-M_mm_per_month.csv')
df_rain_WaPOR = load_governorate_data(rainfall_WaPOR, name='Rainfall_WaPOR')

# --- Load WaPOR Reference ET ---
ret_WaPOR = os.path.join(data_folder, '/content/data/irq_admin1_L1-RET-M_mm_per_month.csv')
df_ret_WaPOR = load_governorate_data(ret_WaPOR, name='RET_WaPOR')

#print(f"\n✅ Combined DataFrame: {df.shape[0]} months × {df.shape[1]} variables")
#print(f"   Period: {df.index.min().strftime('%b %Y')} — {df.index.max().strftime('%b %Y')}")
#print(f"   Missing values: {df.isnull().sum().sum()}")

---
## 1. 📈 Static visualizations
### Comparing WaPOR and AgERA5 climate data (PCP and RET)
Loading PCP and RET data from WaPOR and AgERA5

In [ ]:
# =============================================================================
# CREATING DATAFRAME WITH PCP AND RET data 
# =============================================================================

# Resample monthly rainfall data to annual sums
annual_rain_AgERA = df_rain_AgERA.resample('YE').sum().stack().reset_index()
annual_rain_WaPOR = df_rain_WaPOR.resample('YE').sum().stack().reset_index()

# Rename columns and add source for rainfall
annual_rain_AgERA.columns = ['year', 'governorate', 'rainfall_annual']
annual_rain_AgERA['source'] = 'AgERA5'
annual_rain_WaPOR.columns = ['year', 'governorate', 'rainfall_annual']
annual_rain_WaPOR['source'] = 'WaPOR'

# Combine both sources for rainfall
annual_rainfall_combined = pd.concat([annual_rain_AgERA, annual_rain_WaPOR])
annual_rainfall_combined['year'] = annual_rainfall_combined['year'].dt.year # Extract year as integer

# Resample monthly RET data to annual sums
annual_ret_AgERA = df_ret_AgERA.resample('YE').sum().stack().reset_index()
annual_ret_WaPOR = df_ret_WaPOR.resample('YE').sum().stack().reset_index()

# Rename columns and add source for RET
annual_ret_AgERA.columns = ['year', 'governorate', 'ret_annual']
annual_ret_AgERA['source'] = 'AgERA5'
annual_ret_WaPOR.columns = ['year', 'governorate', 'ret_annual']
annual_ret_WaPOR['source'] = 'WaPOR'

# Combine both sources for RET
annual_ret_combined = pd.concat([annual_ret_AgERA, annual_ret_WaPOR])
annual_ret_combined['year'] = annual_ret_combined['year'].dt.year # Extract year as integer

# Pivot annual rainfall data to have separate columns for AgERA5 and WaPOR
df_rainfall_pivoted = annual_rainfall_combined.pivot_table(
    index=['year', 'governorate'],
    columns='source',
    values='rainfall_annual'
).reset_index()
df_rainfall_pivoted.rename(columns={'AgERA5': 'rainfall_AgERA5', 'WaPOR': 'rainfall_WaPOR'}, inplace=True)

# Pivot annual RET data to have separate columns for AgERA5 and WaPOR
df_ret_pivoted = annual_ret_combined.pivot_table(
    index=['year', 'governorate'],
    columns='source',
    values='ret_annual'
).reset_index()
df_ret_pivoted.rename(columns={'AgERA5': 'RET_AgERA5', 'WaPOR': 'RET_WaPOR'}, inplace=True)

# Merge the pivoted rainfall and RET dataframes
# Use 'outer' merge to keep all years and governorates present in either dataset
final_comparison_df = pd.merge(
    df_rainfall_pivoted,
    df_ret_pivoted,
    on=['year', 'governorate'],
    how='outer'
)

# Calculate difference columns
final_comparison_df['rainfall_difference'] = final_comparison_df['rainfall_WaPOR'] - final_comparison_df['rainfall_AgERA5']
final_comparison_df['RET_difference'] = final_comparison_df['RET_WaPOR'] - final_comparison_df['RET_AgERA5']

# Filter to keep only years where all four columns have data
columns_to_check = ['rainfall_AgERA5', 'rainfall_WaPOR', 'RET_AgERA5', 'RET_WaPOR']
final_comparison_df = final_comparison_df.dropna(subset=columns_to_check)

print("DataFrame with Rainfall and RET from both sources (filtered for complete data) including differences:")
display(final_comparison_df.head())

### Scatter Plot: AgERA5 vs WaPOR Rainfall

This cell creates a **static scatter plot** to visually compare two rainfall datasets — AgERA5 and WaPOR — across different years and governorates in Iraq.

The plot is built using two libraries:
- **Matplotlib** (`plt`): the core plotting engine, used to draw the scatter points, reference line, and text box
- **Seaborn** (`sns`): used here only to generate a color palette, assigning a distinct color to each year

The cell is structured around three main components:

* **The scatter points**: For each year in the dataset, a separate group of points is plotted — where each point represents one governorate. The x-axis shows AgERA5 rainfall and the y-axis shows WaPOR rainfall, so points closer to the diagonal indicate better agreement between the two sources.

* **The 1:1 reference line**: A dashed black line is drawn from the minimum to the maximum value across both datasets. If both datasets were in perfect agreement, all points would fall exactly on this line.

* **The R² text box**: The Pearson correlation coefficient between the two rainfall columns is computed first, then squared to obtain R² — a measure of how well the two datasets agree. An R² close to 1.0 means strong agreement. This value is displayed in a small box inside the plot.


In [ ]:
# =============================================================================
# Scatterplot: AgERA5 vs WaPOR Rainfall - unique color per year
# =============================================================================

# Get unique years from the final_comparison_df
years = sorted(final_comparison_df['year'].unique())

plt.figure(figsize=(10, 8))

# Define a color palette for the years
colors = sns.color_palette('viridis', n_colors=len(years))

for i, year in enumerate(years):
    year_data = final_comparison_df[final_comparison_df['year'] == year]
    plt.scatter(year_data['rainfall_AgERA5'], year_data['rainfall_WaPOR'],
                label=str(year), color=colors[i], alpha=0.7)
corr_rainfall = final_comparison_df['rainfall_AgERA5'].corr(final_comparison_df['rainfall_WaPOR'])
r2_rainfall=corr_rainfall**2

# Add a 1:1 line for reference
min_val = min(final_comparison_df['rainfall_AgERA5'].min(), final_comparison_df['rainfall_WaPOR'].min())
max_val = max(final_comparison_df['rainfall_AgERA5'].max(), final_comparison_df['rainfall_WaPOR'].max())
plt.plot([min_val, max_val], [min_val, max_val], 'k--', lw=2, label='1:1 Line')

# Add R-squared in a text box
plt.text(0.05, 0.95, f'R² = {r2_rainfall:.3f}', transform=plt.gca().transAxes, fontsize=12, verticalalignment='top', bbox=dict(boxstyle='round,pad=0.5', fc='wheat', alpha=0.5))

plt.title('Rainfall: AgERA5 vs WaPOR by Year')
plt.xlabel('AgERA5 Rainfall (mm/year)')
plt.ylabel('WaPOR Rainfall (mm/year)')
plt.legend(title='Year', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

Can you now also create the same graph for RET? 

In [ ]:
# =============================================================================
# Scatterplot: AgERA5 vs WaPOR Reference ET - unique color per year
# =============================================================================



This cell creates a scatter plot comparing **Reference Evapotranspiration (RET)** from AgERA5 and WaPOR,
with each **governorate** represented by a distinct color using the `tab20` palette —
suitable when you have many categories (up to 20). Like the previous plot, it includes a **1:1 reference line**
and an **R² value** to assess agreement between the two datasets.

> 🎨 To explore other Matplotlib color palettes, see the official guide:
> [Matplotlib Colormap Reference](https://matplotlib.org/stable/gallery/color/colormap_reference.html)

In [ ]:
# =============================================================================
# Scatterplot: AgERA5 vs WaPOR Reference ET - unique color per governorate
# =============================================================================

# Get unique governorates from the final_comparison_df
governorates = sorted(final_comparison_df['governorate'].unique())

plt.figure(figsize=(10, 8))

# Define a color palette for the governorates
colors = sns.color_palette('tab20', n_colors=len(governorates)) # Using 'tab20' for a good number of distinct colors

for i, gov in enumerate(governorates):
    gov_data = final_comparison_df[final_comparison_df['governorate'] == gov]
    plt.scatter(gov_data['RET_AgERA5'], gov_data['RET_WaPOR'],
                label=gov, color=colors[i], alpha=0.7)

# Calculate R-squared for RET data
corr_ret = final_comparison_df['RET_AgERA5'].corr(final_comparison_df['RET_WaPOR'])
r2_ret = corr_ret**2

# Add a 1:1 line for reference
min_val = min(final_comparison_df['RET_AgERA5'].min(), final_comparison_df['RET_WaPOR'].min())
max_val = max(final_comparison_df['RET_AgERA5'].max(), final_comparison_df['RET_WaPOR'].max())
plt.plot([min_val, max_val], [min_val, max_val], 'k--', lw=2, label='1:1 Line')

# Add R-squared in a text box
plt.text(0.05, 0.95, f'R² = {r2_ret:.3f}', transform=plt.gca().transAxes, fontsize=12, verticalalignment='top', bbox=dict(boxstyle='round,pad=0.5', fc='wheat', alpha=0.5))

plt.title('RET: AgERA5 vs WaPOR by Governorate')
plt.xlabel('AgERA5 RET (mm/year)')
plt.ylabel('WaPOR RET (mm/year)')
plt.legend(title='Governorate', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

---

## 👇 2. Interactive visualizations
### Bokeh Building Blocks

On the most general level, all Bokeh visualizations are called **documents**.
A Bokeh document can contain lots of different elements. The most common
building blocks of a Bokeh document are:

#### Plots

The most common element in a Bokeh visualization is a plot. A plot is a
graphical representation of data. It consists of elements like glyphs, axes,
legends, and annotations:

Bokeh plots can also contain interactive tools in a toolbar. By default, the
following tools are included:

<table>
<tr><td><img src="https://www.clipartmax.com/png/small/437-4377183_four-grouped-arrows-button-to-move-svg-png-icon-free-move-icon.png"height="30" width="auto" align="left"></td><td>Pan: Pan the plot horizontally and vertically.</td></tr>
<tr><td><img src="https://www.svgrepo.com/show/340519/zoom-in-area.svg" height="30" width="auto"align="left"></td><td>Box Zoom: Zoom in and out of the plot by selecting a rectangular area.</td></tr>
<tr><td><img src="https://png.pngtree.com/png-clipart/20191122/original/pngtree-zoom-icon-isolated-on-abstract-background-png-image_5192365.jpg" height="30" width="auto" align="left"></td><td>Wheel Zoom: Zoom in and out of the plot using the mouse wheel.</td></tr>
<tr><td><img src="https://static.thenounproject.com/png/3974682-200.png" height="35" width="auto"align="left"></td><td>Save: Save the plot as a PNG file.</td></tr>
<tr><td><img src="https://spng.pngfind.com/pngs/s/429-4296335_png-file-refresh-icon-svg-transparent-png.png" height="25" width="auto" align="left"></td><td>Reset: Reset the plot to its original state.</td></tr>
<tr><td><img src="https://www.svgrepo.com/show/49680/round-help-button.svg" height="28" width="auto"lign="left"></td><td>Help: Opens a help page with information about the tools available in Bokeh.</td></tr>
</table>

### Interactive RET: AgERA5 vs WaPOR by Governorate (with 👆  Hover Highlighting)

This enhanced interactive scatter plot displays the annual Reference Evapotranspiration (RET) from AgERA5 against WaPOR for different governorates and years.

**New Interactive Features:**

*   **Hover Highlighting:** When you hover your cursor over a data point, all other points from the *same governorate* will become fully opaque (highlighted), while points from other governorates will become semi-transparent.
*   **Hover Details:** Moving your cursor over any point will display a tooltip with the Governorate, Year, AgERA5 RET, and WaPOR RET values.
*   **Legend Interactivity:** Click on a governorate in the legend to hide/show its data points on the plot. To reset, double-click a legend item.
*   **Zoom and Pan:** Use the tools in the plot's toolbar to zoom in/out and pan across the plot.

Explore an example of the interactive tools available in Bokeh
<details>
<summary style="cursor:pointer; color:#555; font-size:18px; margin-top:10px;"><b>Example </b></summary>

<br>

![My GIF](https://miro.medium.com/v2/0*lfsR26JXj4o_QMWI.gif)
</details>

In [ ]:
# =============================================================================
# Create interactive RET graph using Bokeh
# =============================================================================

# ── DATA SOURCE ───────────────────────────────────────────────────────────
source = ColumnDataSource(data={
    'governorate': final_comparison_df['governorate'].tolist(),
    'RET_AgERA5': final_comparison_df['RET_AgERA5'].tolist(),
    'RET_WaPOR': final_comparison_df['RET_WaPOR'].tolist()
})

# ── COLOR MAPPING ─────────────────────────────────────────────────────────
governorate_list = sorted(final_comparison_df['governorate'].unique().tolist())
palette = Category20[len(governorate_list)]

# ── CREATE FIGURE ─────────────────────────────────────────────────────────
p = figure(title='RET: AgERA5 vs WaPOR by Governorate 2018-2025',
           height=600, width=800,
           x_axis_label='AgERA5 RET (mm/year)',
           y_axis_label='WaPOR RET (mm/year)',
           tools="pan,wheel_zoom,reset,save,box_zoom, hover")

p.title.text_font_size = '11pt' # Set title font size after creating the figure

# ── SCATTER ───────────────────────────────────────────────────────────────
scatter = p.scatter(x='RET_AgERA5', y='RET_WaPOR', source=source,
                    legend_field='governorate', size=8, alpha=0.7,
                    color=factor_cmap('governorate', palette=palette, factors=governorate_list),
                    hover_color="red")

# Increase the size of the point on hover
scatter.hover_glyph.size = 15

# ── HOVER TOOL ────────────────────────────────────────────────────────────
hover = HoverTool(renderers=[scatter],
                  tooltips=[
                      ("Governorate", "@governorate"),
                      ("AgERA5 RET", "@RET_AgERA5{0.00} mm"),
                      ("WaPOR RET", "@RET_WaPOR{0.00} mm")
                  ])
p.add_tools(hover)

# ── 1:1 LINE ──────────────────────────────────────────────────────────────
min_val = final_comparison_df[['RET_AgERA5','RET_WaPOR']].min().min()
max_val = final_comparison_df[['RET_AgERA5','RET_WaPOR']].max().max()
p.line([min_val, max_val], [min_val, max_val],
       line_color='red', line_dash='dashed', line_width=2, legend_label='1:1 Line')

# ── LEGEND ────────────────────────────────────────────────────────────────
p.add_layout(p.legend[0], 'right')


# ── R² LABEL ──────────────────────────────────────────────────────────────

corr = final_comparison_df['RET_AgERA5'].corr(final_comparison_df['RET_WaPOR'])
r2 = corr**2

min_val = final_comparison_df[['RET_AgERA5','RET_WaPOR']].min().min()
max_val = final_comparison_df[['RET_AgERA5','RET_WaPOR']].max().max()

r2_label = Label(
    x=min_val * 0.95,
    y=max_val * 0.95,
    text_align='left', text_baseline='top',
    text=f'R² = {r2:.3f}',
    text_font_size='11pt', text_color='black',
    background_fill_color='wheat',
    background_fill_alpha=0.8,
    border_line_color='black',
    border_line_width=1,
    padding=5
)
p.add_layout(r2_label)

show(p)


📎 Now that we have built an interactive scatter plot, we can take it further by organizing
multiple plots into **tabs** and **hover** — allowing the user to switch between different views
(e.g. **all years combined** or **year by year** ) without scrolling through a long notebook.

<img src="https://user-images.githubusercontent.com/38312200/81195451-1f4e3a00-8fbe-11ea-8909-5876dd312326.gif" height="500" width="auto" align="left">



In [ ]:
# =============================================================================
# Adding tabs to interactive RET graph using Bokeh
# =============================================================================

# ── FULL DATA SOURCE ───────────────────────────────────────────────────────
# Store the entire dataset in a Bokeh ColumnDataSource.
# This is used by the JavaScript callback to filter data when the year changes, without needing to go back to Python.
full_data_source_all_years = ColumnDataSource(data={
    'year': final_comparison_df['year'].tolist(),
    'governorate': final_comparison_df['governorate'].tolist(),
    'RET_AgERA5': final_comparison_df['RET_AgERA5'].tolist(),
    'RET_WaPOR': final_comparison_df['RET_WaPOR'].tolist()
})

# ──  INITIAL FILTER ─────────────────────────────────────────────────────────
# Select the earliest year as the default view when the plot first loads.
initial_year = sorted(final_comparison_df['year'].unique().tolist())[0]
initial_df_filtered = final_comparison_df[final_comparison_df['year'] == initial_year].copy()

# ──  DISPLAY DATA SOURCE ────────────────────────────────────────────────────
# This is the data source actually connected to the plot glyphs.
# It starts with the initial year's data and gets updated by the JS callback when the user selects a different year. 
source_ret_display = ColumnDataSource(data={
    'year': initial_df_filtered['year'],
    'governorate': initial_df_filtered['governorate'],
    'RET_AgERA5': initial_df_filtered['RET_AgERA5'],
    'RET_WaPOR': initial_df_filtered['RET_WaPOR'],
    'alpha_value': [0.7] * len(initial_df_filtered) # controls point transparency — used for hover highlighting.
})

# ──  COLOR MAPPING ──────────────────────────────────────────────────────────
# Assign a distinct color to each governorate using the Category20 palette.
governorate_list = sorted(final_comparison_df['governorate'].unique().tolist())
num_governorates = len(governorate_list)
palette = Category20[num_governorates] if num_governorates <= 20 else Category20[20] 

# ──  AXIS RANGE ─────────────────────────────────────────────────────────────
# Calculate the global min/max across both datasets to keep axis ranges consistent when switching between years.
min_val_ret = min(final_comparison_df['RET_AgERA5'].min(), final_comparison_df['RET_WaPOR'].min())
max_val_ret = max(final_comparison_df['RET_AgERA5'].max(), final_comparison_df['RET_WaPOR'].max())

# ──  CREATE FIGURE ──────────────────────────────────────────────────────────
# Define the main Bokeh figure with axis labels, dimensions, and interactive tools.
p_ret_interactive = figure(title=f'Interactive RET: AgERA5 vs WaPOR by Governorate (Year: {initial_year})',
                           height=600, width=1000,
                           x_axis_label='AgERA5 RET (mm/year)',
                           y_axis_label='WaPOR RET (mm/year)',
                           tools="pan,wheel_zoom,box_zoom,reset,save",
                           x_range=Range1d(min_val_ret * 0.9, max_val_ret * 1.1),
                           y_range=Range1d(min_val_ret * 0.9, max_val_ret * 1.1))

# ── SCATTER GLYPHS ─────────────────────────────────────────────────────────
# Plot each data point as a circle. Color is mapped by governorate using factor_cmap, and transparency is driven by 'alpha_value' for hover effects.
scatter_renderer = p_ret_interactive.scatter(x='RET_AgERA5', y='RET_WaPOR', source=source_ret_display,
                          legend_field='governorate', size=8, alpha='alpha_value',
                          color=factor_cmap('governorate', palette=palette, factors=governorate_list))


# ── FONT SIZES ─────────────────────────────────────────────────────────────
p_ret_interactive.title.text_font_size = "15pt"
p_ret_interactive.xaxis.axis_label_text_font_size = "13pt"
p_ret_interactive.yaxis.axis_label_text_font_size = "13pt"
p_ret_interactive.xaxis.major_label_text_font_size = "11pt"
p_ret_interactive.yaxis.major_label_text_font_size = "11pt"

# ──  HOVER CALLBACK (JavaScript) ───────────────────────────────────────────
# When the user hovers over a point, all points from the same governorate become fully opaque (alpha=1.0), while others are faded (alpha=0.2). 
# When no point is hovered, all points return to the default alpha (0.7).
hover_callback_code = """
    const data = source.data;
    const governorates = data['governorate'];
    const current_alpha = data['alpha_value'];
    const hovered_indices = cb_data.index.indices;

    if (hovered_indices.length > 0) {
        const hovered_governorate = governorates[hovered_indices[0]];
        for (let i = 0; i < governorates.length; i++) {
            current_alpha[i] = (governorates[i] === hovered_governorate) ? 1.0 : 0.2;
        }
    } else {
        for (let i = 0; i < governorates.length; i++) {
            current_alpha[i] = 0.7;
        }
    }
    source.change.emit(); // Tell Bokeh the data has changed so it re-renders
"""

hover_callback = CustomJS(args={'source': source_ret_display}, code=hover_callback_code)

# ──  HOVER TOOL ────────────────────────────────────────────────────────────
# Configure the tooltip content shown on hover and attach the highlight callback.
hover = HoverTool(renderers=[scatter_renderer],
                  tooltips="""
                    <div style="font-size:14px; padding:5px;">
                        <b style="font-size:15px;">@governorate</b><br>
                        <span>Year: @year</span><br>
                        <span>AgERA5 RET: @RET_AgERA5{0.00} mm</span><br>
                        <span>WaPOR RET: @RET_WaPOR{0.00} mm</span>
                    </div>
                  """,
                  callback=hover_callback)
p_ret_interactive.add_tools(hover)

# ── .1:1 REFERENCE LINE ───────────────────────────────────────────────────
# A dashed red line where AgERA5 = WaPOR (perfect agreement).
p_ret_interactive.line([min_val_ret, max_val_ret], [min_val_ret, max_val_ret],
                       line_color='red', line_dash='dashed', line_width=2, legend_label='1:1 Line')

# ──  R² CALCULATION ────────────────────────────────────────────────────────
corr_ret_initial = initial_df_filtered['RET_AgERA5'].corr(initial_df_filtered['RET_WaPOR'])
r2_ret_initial = corr_ret_initial**2 if not pd.isna(corr_ret_initial) else float('nan')
r2_initial_text = f'R² = {r2_ret_initial:.3f}' if not pd.isna(r2_ret_initial) else 'R² = N/A'

corr_ret_all_years = final_comparison_df['RET_AgERA5'].corr(final_comparison_df['RET_WaPOR'])
r2_ret_all_years = corr_ret_all_years**2 if not pd.isna(corr_ret_all_years) else float('nan')
r2_all_years_text = f'R² = {r2_ret_all_years:.3f}' if not pd.isna(r2_ret_all_years) else 'R² = N/A'

# ──  R² LABEL (inside the plot) ───────────────────────────────────────────
# A text box displayed inside the plot showing the current R² value.
# Positioned in the upper-left corner using data coordinates. 
r2_label_interactive = Label(
    x=min_val_ret * 0.95,
    y=max_val_ret * 1.05,
    text_align='left', text_baseline='top',
    text=r2_initial_text,
    text_font_size='11pt', text_color='black',
    background_fill_color='wheat',
    background_fill_alpha=0.8,
    border_line_color='black',
    border_line_width=1,
    padding=5
)
p_ret_interactive.add_layout(r2_label_interactive)

# ──  LEGEND CONFIGURATION ──────────────────────────────────────────────────
p_ret_interactive.add_layout(p_ret_interactive.legend[0], 'right')
p_ret_interactive.legend.orientation = "vertical"
p_ret_interactive.legend.border_line_width = 1
p_ret_interactive.legend.border_line_color = "black"
p_ret_interactive.legend.background_fill_alpha = 0.8
p_ret_interactive.legend.spacing = 5
p_ret_interactive.legend.label_text_font_size = "11pt"
p_ret_interactive.legend.label_text_font_style = "bold"  # <--- Added bold style for the entire legend!

# ──  YEAR SELECTOR WIDGET ──────────────────────────────────────────────────
# A dropdown menu allowing the user to filter the plot by year.
year_options = ['All Years'] + [str(y) for y in sorted(final_comparison_df['year'].unique().tolist())]
year_select = Select(title='Select Year:', value=str(initial_year), options=year_options)

# ── 16. YEAR SELECTION CALLBACK (JavaScript) ─────────────────────────────────
# Triggered when the user changes the year in the dropdown.
# Filters the full dataset client-side (no Python needed) and updates:
#   - the displayed scatter points
#   - the plot title
#   - the R² label (recalculated in JS for individual years)
year_callback_code = """
    const selected_year_str = year_select.value;
    const full_data = full_data_source_all_years.data;
    const display_data = source_ret_display.data;

    const new_year_values = [];
    const new_governorate_values = [];
    const new_agera_values = [];
    const new_wapor_values = [];
    const new_alpha_values = [];

    if (selected_year_str === 'All Years') {
        for (let i = 0; i < full_data['year'].length; i++) {
            new_year_values.push(full_data['year'][i]);
            new_governorate_values.push(full_data['governorate'][i]);
            new_agera_values.push(full_data['RET_AgERA5'][i]);
            new_wapor_values.push(full_data['RET_WaPOR'][i]);
            new_alpha_values.push(0.7);
        }
        p_ret_interactive.title.text = `Interactive RET: AgERA5 vs WaPOR by Governorate (All Years)`;
        r2_label_interactive.text = r2_all_years_text; // Use pre-calculated value
    } else {
        const selected_year = parseInt(selected_year_str);
        for (let i = 0; i < full_data['year'].length; i++) {
            if (full_data['year'][i] === selected_year) {
                new_year_values.push(full_data['year'][i]);
                new_governorate_values.push(full_data['governorate'][i]);
                new_agera_values.push(full_data['RET_AgERA5'][i]);
                new_wapor_values.push(full_data['RET_WaPOR'][i]);
                new_alpha_values.push(0.7);
            }
        }
        p_ret_interactive.title.text = `Interactive RET: AgERA5 vs WaPOR by Governorate (Year: ${selected_year})`;

        // Recalculate R² in JavaScript using Pearson correlation formula
        let r2_val = 'N/A';
        if (new_agera_values.length > 1) {
            let sum_x = 0, sum_y = 0, sum_xy = 0, sum_x2 = 0, sum_y2 = 0;
            const n = new_agera_values.length;
            for (let i = 0; i < n; i++) {
                const x = new_agera_values[i];
                const y = new_wapor_values[i];
                sum_x += x; sum_y += y; sum_xy += x * y;
                sum_x2 += x * x; sum_y2 += y * y;
            }
            const numerator = (n * sum_xy) - (sum_x * sum_y);
            const denominator_x = (n * sum_x2) - (sum_x * sum_x);
            const denominator_y = (n * sum_y2) - (sum_y * sum_y);
            if (denominator_x > 0 && denominator_y > 0) {
                const r = numerator / Math.sqrt(denominator_x * denominator_y);
                r2_val = (r * r).toFixed(3);
            }
        }
        r2_label_interactive.text = `R² = ${r2_val}`;
    }

    // Push the filtered data into the display source and trigger re-render
    display_data['year'] = new_year_values;
    display_data['governorate'] = new_governorate_values;
    display_data['RET_AgERA5'] = new_agera_values;
    display_data['RET_WaPOR'] = new_wapor_values;
    display_data['alpha_value'] = new_alpha_values;
    source_ret_display.change.emit();
"""

# ──  CONNECT CALLBACK TO WIDGET ───────────────────────────────────────────
# Pass all required Python objects to the JS callback and link it to the widget.
year_callback = CustomJS(args={
    'year_select': year_select,
    'full_data_source_all_years': full_data_source_all_years,
    'source_ret_display': source_ret_display,
    'p_ret_interactive': p_ret_interactive,
    'r2_label_interactive': r2_label_interactive,
    'r2_all_years_text': r2_all_years_text
}, code=year_callback_code)
year_select.js_on_change('value', year_callback)

# ──  LAYOUT & DISPLAY ──────────────────────────────────────────────────────
# Stack the year selector above the plot and render everything in the notebook.
layout = column(year_select, p_ret_interactive)
show(layout)


> 💡 **Try it yourself with 🤖 Gemini!**
>
> You don't need to know JavaScript or programming to customize this interactive plot.
> You can use **Gemini** to help you modify it.
> Just describe what you want to change in plain English.
>
> **Example prompt you can try:**
>
> *"I have a Bokeh interactive scatter plot in Python with a year selector dropdown
> and a hover tooltip. Can you help me:*
> - *Change the point size from 8 to 12*
> - *Add the number of points (n) to the R² label so it shows `R² = 0.992 (n = 19)`*
> - *Change the 1:1 line color from red to black*
> - *Add a second dropdown to filter by governorate*"*
>
> The more specific you are about what you want to change and where it is in the code,
> the better the result. You can copy the cell above and paste it directly into the chat!
>
> ⚠️ **Before you apply any changes:**
> Always ask Gemini to **comment every modified line** so you know exactly what was changed.
> For example, add to your prompt:
> *"Please add a comment on each line you modify, explaining what you changed and why."*
> This makes it much easier to understand, debug, and revert changes if something breaks.
>
><img src="https://images.fonearena.com/blog/wp-content/uploads/2026/04/Google-Colab-Learn-Mode.gif" height="250" width="auto" align="center">

## 3. Interactive Dashboard

Lets create a complete dashboard for PCP and RET, including a map showing the data in a spatial manner.  

<p style="background-color: #FFCCCC; padding: 10px;">
👇 Run the code cell below first to initialize some of the variables used for the dashboard you'll be building:
</p>

In [ ]:
# =============================================================================
# INITIALISE VARIABLES FOR DASHBOARD
# =============================================================================

# =============================================================================
# LOAD SHAPEFILE
# =============================================================================
iraq_gdf = gpd.read_file('/content/data/irq_admin1.geojson')
iraq_gdf = iraq_gdf[['adm1_name', 'geometry']].rename(columns={'adm1_name': 'governorate'})

# =============================================================================
# INITIAL VARIABLES
# =============================================================================
initial_year = sorted(final_comparison_df['year'].unique().tolist())[0]
all_years = sorted(final_comparison_df['year'].unique().tolist())
year_options = [str(y) for y in all_years]
initial_map_source = 'AgERA5'
initial_data_type = 'Rainfall'
initial_governorate_for_scatter = sorted(final_comparison_df['governorate'].unique().tolist())[0]
governorate_options = sorted(final_comparison_df['governorate'].unique().tolist())

global_min_rainfall = min(final_comparison_df['rainfall_AgERA5'].min(), final_comparison_df['rainfall_WaPOR'].min())
global_max_rainfall = max(abs(final_comparison_df['rainfall_AgERA5'].max()), abs(final_comparison_df['rainfall_WaPOR'].max()))
global_min_ret = min(final_comparison_df['RET_AgERA5'].min(), final_comparison_df['RET_WaPOR'].min())
global_max_ret = max(abs(final_comparison_df['RET_AgERA5'].max()), abs(final_comparison_df['RET_WaPOR'].max()))

# =============================================================================
# BUILD all_dashboard_data
# Nested structure: {governorate: {year: {'Rainfall': {'AgERA5': val, 'WaPOR': val},
#                                         'RET':      {'AgERA5': val, 'WaPOR': val}}}}
# =============================================================================
all_dashboard_data = {}

for gov in governorate_options:
    all_dashboard_data[gov] = {}
    for year in all_years:
        # Renamed 'row' DataFrame variable to 'filtered_row' to avoid conflict with bokeh.layouts.row
        filtered_row = final_comparison_df[
            (final_comparison_df['governorate'] == gov) &
            (final_comparison_df['year'] == year)
        ]
        if not filtered_row.empty:
            all_dashboard_data[gov][year] = {
                'Rainfall': {
                    'AgERA5': float(filtered_row['rainfall_AgERA5'].values[0]),
                    'WaPOR':  float(filtered_row['rainfall_WaPOR'].values[0])
                },
                'RET': {
                    'AgERA5': float(filtered_row['RET_AgERA5'].values[0]),
                    'WaPOR':  float(filtered_row['RET_WaPOR'].values[0])
                }
            }

# =============================================================================
# BUILD all_map_data_for_js
# Nested structure: {year: {source: {data_type: {'geojson': geojson_string}}}}
# The 'value' column drives the color mapper in the map.
# All four hover columns are kept intact to avoid ??? in tooltips.
# =============================================================================
all_map_data_for_js = {}

data_type_col_map = {
    'AgERA5':     {'Rainfall': 'rainfall_AgERA5', 'RET': 'RET_AgERA5'},
    'WaPOR':      {'Rainfall': 'rainfall_WaPOR',  'RET': 'RET_WaPOR'},
    'Difference': {'Rainfall': 'rainfall_difference', 'RET': 'RET_difference'}
}

for year in all_years:
    all_map_data_for_js[str(year)] = {}
    year_df = final_comparison_df[final_comparison_df['year'] == year]

    for source in ['AgERA5', 'WaPOR', 'Difference']:
        all_map_data_for_js[str(year)][source] = {}

        for data_type in ['Rainfall', 'RET']:
            col = data_type_col_map[source][data_type]

            # Always include the four hover columns explicitly
            merge_cols = ['governorate', 'rainfall_AgERA5', 'rainfall_WaPOR',
                          'RET_AgERA5', 'RET_WaPOR']

            # Add col only if it is not already in the list (e.g. Difference columns)
            if col not in merge_cols:
                merge_cols.append(col)

            merged = iraq_gdf.merge(
                year_df[merge_cols],
                on='governorate',
                how='left'
            )

            # Copy the selected column into 'value' — keeps original columns intact
            merged['value'] = merged[col]

            # Drop any accidental duplicate columns
            merged = merged.loc[:, ~merged.columns.duplicated()]

            geojson_str = merged.to_json()
            all_map_data_for_js[str(year)][source][data_type] = {
                'geojson': geojson_str
            }

print("✅ all_dashboard_data built:", len(all_dashboard_data), "governorates")
print("✅ all_map_data_for_js built:", len(all_map_data_for_js), "years")
print("✅ Years:", all_years)
print("✅ Global ranges:")
print(f"   Rainfall: {global_min_rainfall:.1f} - {global_max_rainfall:.1f} mm")
print(f"   RET:      {global_min_ret:.1f} - {global_max_ret:.1f} mm")

## **Dashboard Iraq Climate Data Dashboard (WaPOR vs AgERA5)**
---
### 📊 Dashboard Overview

This cell builds a fully interactive Bokeh dashboard to compare WaPOR and AgERA5 climate data across Iraq. It consists of four main components:

1. **Interactive Widgets:** Dropdowns and radio buttons allow you to filter the data by Year, Data Type (Rainfall or RET), Map Source (AgERA5, WaPOR, or Difference), and Governorate
2. **Choropleth Map:** Displays the spatial distribution of the selected data. It dynamically updates its color palette: *Blues* for Rainfall, *Greens* for RET, and a divergent *Red-Blue* scale when visualizing the "Difference" between datasets.
3. **Scatterplot:** Compares the selected datasets for a specific governorate across all years. It includes a 1:1 reference line and dynamically recalculates the $R^2$ correlation on the fly.
4. **CustomJS Callback:** The "brain" of the dashboard. A block of JavaScript code ensures that whenever you interact with a widget, the map, plot, colors, and statistics update instantly right in your browser, without needing to rerun any Python code!

In [ ]:
# =============================================================================
# CREATE DASHBOARD
# =============================================================================

# Calculate maximum absolute differences for symmetric color scaling
global_max_diff_rainfall = max(abs(final_comparison_df['rainfall_difference'].min()), abs(final_comparison_df['rainfall_difference'].max()))
global_max_diff_ret = max(abs(final_comparison_df['RET_difference'].min()), abs(final_comparison_df['RET_difference'].max()))

# =============================================================================
# WIDGETS
# =============================================================================

# Year Selector
year_select = Select(title='Select Year:', value=str(initial_year), options=year_options)
# Data Type Radio Buttons
data_type_title = Div(text="<span style='font-size: 11px; font-weight: bold; color: #444;'>Select Data Type:</span>", margin=(5, 5, 0, 5))
data_type_buttons = RadioButtonGroup(labels=['Rainfall', 'RET'], active=0, margin=(5, 5, 5, 5))
data_type_box = column(data_type_title, data_type_buttons)

# Map Source Selector
map_source_options = ['AgERA5', 'WaPOR', 'Difference']
map_source_select = Select(title='Select Map Data Source:', value=initial_map_source, options=map_source_options)

# Governorate Selector (for Scatterplot)
governorate_select = Select(title='Select Governorate (for Scatterplot):', value=initial_governorate_for_scatter, options=governorate_options)

# =============================================================================
# MAP SETUP
# =============================================================================

# LinearColorMapper maps numeric values to colors —
# Blues256[::-1] reverses the palette so dark blue = high values

color_mapper = LinearColorMapper(
    palette=Blues256[::-1],
    low=global_min_rainfall,
    high=global_max_rainfall,
    nan_color='#d9d9d9'
)

# GeoJSONDataSource feeds the map polygons —
# updated by the JS callback when year/source/data type changes

initial_map_geojson_source = GeoJSONDataSource(
    geojson=all_map_data_for_js[str(initial_year)][initial_map_source][initial_data_type]['geojson']
)

p_map = figure(
    title=f'{initial_map_source} Annual {initial_data_type} ({initial_year})',
    height=600, width=600,
    tools="pan,wheel_zoom,box_zoom,reset,save",
    match_aspect=False
)

p_map.patches(
    xs='xs', ys='ys',
    source=initial_map_geojson_source,
    fill_color={'field': 'value', 'transform': color_mapper},
    line_color='white',
    line_width=0.5
)

color_bar = ColorBar(color_mapper=color_mapper, width=8, location=(0, 0))
p_map.add_layout(color_bar, 'right')

map_hover = HoverTool(tooltips=[
    ('Governorate',     '@governorate'),
    ('Rainfall AgERA5', '@rainfall_AgERA5{0.00} mm'),
    ('Rainfall WaPOR',  '@rainfall_WaPOR{0.00} mm'),
    ('RET AgERA5',      '@RET_AgERA5{0.00} mm'),
    ('RET WaPOR',       '@RET_WaPOR{0.00} mm'),
])
p_map.add_tools(map_hover)
p_map.title.text_font_size = "13pt"

# =============================================================================
# SCATTERPLOT SETUP
# =============================================================================

# Initial scatterplot data (AgERA5 vs WaPOR for a selected governorate across years)
initial_scatter_years = []
initial_scatter_agera_values = []
initial_scatter_wapor_values = []

# Populate initial scatter data based on initial_governorate_for_scatter and initial_data_type
# (initial_data_type is 'Rainfall' by default)
for year in all_years:
  # Ensure data exists for the initial governorate and data type
    gov_data_for_year = all_dashboard_data[initial_governorate_for_scatter].get(year, {}).get(initial_data_type, {})
    if 'AgERA5' in gov_data_for_year and 'WaPOR' in gov_data_for_year:
        initial_scatter_years.append(year)
        initial_scatter_agera_values.append(gov_data_for_year['AgERA5'])
        initial_scatter_wapor_values.append(gov_data_for_year['WaPOR'])

scatter_source = ColumnDataSource(data=dict(
    year=initial_scatter_years,
    x_val=initial_scatter_agera_values,
    y_val=initial_scatter_wapor_values
))

# Recalculate initial R-squared for Python side based on this new data
initial_r = np.nan
initial_r2 = np.nan
if len(initial_scatter_agera_values) > 1:
    temp_df = pd.DataFrame({'agera': initial_scatter_agera_values, 'wapor': initial_scatter_wapor_values}).dropna()
    if len(temp_df) > 1:
        initial_r, _ = pearsonr(temp_df['agera'], temp_df['wapor'])
        initial_r2 = initial_r**2

initial_r2_text = f"R² = {initial_r2:.3f}" if not np.isnan(initial_r2) else "R² = N/A"

# Determine initial 1:1 line range based on initial_data_type
if initial_data_type == 'Rainfall':
    line_min = global_min_rainfall
    line_max = global_max_rainfall
else: #RET
    line_min = global_min_ret
    line_max = global_max_ret


p_scatter = figure(
    title=f'AgERA5 vs WaPOR Annual {initial_data_type} for {initial_governorate_for_scatter}',
    height=580, width=600,
    x_axis_label='AgERA5 (mm)',
    y_axis_label='WaPOR (mm)',
    tools="pan,wheel_zoom,box_zoom,reset,save",
    x_range=Range1d(line_min, line_max), # Set initial x_range
    y_range=Range1d(line_min, line_max) # Set initial y_range
)

p_scatter.scatter(x='x_val', y='y_val', source=scatter_source, size=8, alpha=0.6)

#Scatter plot in tools
scatter_hover = HoverTool(tooltips=[
    ('Year',   '@year'),
    ('AgERA5', '@x_val{0.00} mm'),
    ('WaPOR',  '@y_val{0.00} mm')
])
p_scatter.add_tools(scatter_hover)

# Add 1:1 line for scatterplot using the determined global min/max
p_scatter.line([line_min, line_max], [line_min, line_max],
               line_color='red', line_dash='dashed', line_width=2)
# R-squared Label
r2_label = Label(
    x=500, y=500, x_units='screen', y_units='screen',
    x_offset=-10, y_offset=-10,
    text_align='right', text_baseline='top',
    text=initial_r2_text,
    text_font_size='12pt', text_color='black',
    background_fill_color='wheat', background_fill_alpha=0.5
)
p_scatter.add_layout(r2_label)

# =============================================================================
# JAVASCRIPT CALLBACK (CustomJS) for Interactivity
# =============================================================================
callback_code = """
    const year = String(parseInt(year_select.value));
    const data_type_index = data_type_buttons.active;
    const data_type = data_type_index === 0 ? 'Rainfall' : 'RET';
    const map_source = map_source_select.value;
    const selected_governorate = governorate_select.value;

    const all_map_data = all_map_data_for_js_callback;
    const all_dashboard_data_for_scatterplot = all_dashboard_data_js;

    // Update color mapper range and palette based on map source
    if (map_source === 'Difference') {
        color_mapper.palette = rdbu_palette;
        if (data_type === 'Rainfall') {
            color_mapper.low = -global_max_diff_rainfall;
            color_mapper.high = global_max_diff_rainfall;
        } else {
            color_mapper.low = -global_max_diff_ret;
            color_mapper.high = global_max_diff_ret;
        }
    } else {
        if (data_type === 'Rainfall') {
            color_mapper.palette = blues_palette;
            color_mapper.low = global_min_rainfall;
            color_mapper.high = global_max_rainfall;
        } else {
            color_mapper.palette = greens_palette;
            color_mapper.low = global_min_ret;
            color_mapper.high = global_max_ret;
        }
    }

    // Update map GeoJSON
    const current_year_map_data = all_map_data[year];
    if (current_year_map_data) {
        const current_source_map_data = current_year_map_data[map_source];
        if (current_source_map_data) {
            const current_data_type_map_data = current_source_map_data[data_type];
            if (current_data_type_map_data) {
                map_source_bokeh.geojson = current_data_type_map_data.geojson;
                p_map.title.text = `${map_source} Annual ${data_type} (${year})`;
            }
        }
    }
    map_source_bokeh.change.emit();
    color_mapper.change.emit();

    // Update scatterplot data
    const new_scatter_years = [];
    const new_scatter_agera_values = [];
    const new_scatter_wapor_values = [];

    const gov_scatter_data_full = all_dashboard_data_for_scatterplot[selected_governorate];

    if (gov_scatter_data_full instanceof Map) {
        const sorted_years = Array.from(gov_scatter_data_full.keys()).sort((a, b) => parseInt(a) - parseInt(b));
        for (const year_key of sorted_years) {
            const year_data_for_gov = gov_scatter_data_full.get(year_key);
            if (year_data_for_gov && year_data_for_gov[data_type]) {
                const agera_val = year_data_for_gov[data_type]['AgERA5'];
                const wapor_val = year_data_for_gov[data_type]['WaPOR'];
                if (agera_val !== undefined && agera_val !== null && !isNaN(agera_val) &&
                    wapor_val !== undefined && wapor_val !== null && !isNaN(wapor_val)) {
                    new_scatter_years.push(parseInt(year_key));
                    new_scatter_agera_values.push(agera_val);
                    new_scatter_wapor_values.push(wapor_val);
                }
            }
        }
    } else if (typeof gov_scatter_data_full === 'object' && gov_scatter_data_full !== null) {
        const sorted_years = Array.from(Object.keys(gov_scatter_data_full)).sort((a, b) => parseInt(a) - parseInt(b));
        for (const year_key of sorted_years) {
            const year_data_for_gov = gov_scatter_data_full[year_key];
            if (year_data_for_gov && year_data_for_gov[data_type]) {
                const agera_val = year_data_for_gov[data_type]['AgERA5'];
                const wapor_val = year_data_for_gov[data_type]['WaPOR'];
                if (agera_val !== undefined && agera_val !== null && !isNaN(agera_val) &&
                    wapor_val !== undefined && wapor_val !== null && !isNaN(wapor_val)) {
                    new_scatter_years.push(parseInt(year_key));
                    new_scatter_agera_values.push(agera_val);
                    new_scatter_wapor_values.push(wapor_val);
                }
            }
        }
    }

    // Update scatter data, title, and axis label Divs
    scatter_source.data = { year: new_scatter_years, x_val: new_scatter_agera_values, y_val: new_scatter_wapor_values };
    p_scatter.title.text = `AgERA5 vs WaPOR Annual ${data_type} for ${selected_governorate}`;

    // Recalculate R²
    let r2_val_scatter = 'N/A';
    if (new_scatter_agera_values.length > 1) {
        const valid_data = [];
        for (let i = 0; i < new_scatter_agera_values.length; i++) {
            if (!isNaN(new_scatter_agera_values[i]) && !isNaN(new_scatter_wapor_values[i])) {
                valid_data.push({ x: new_scatter_agera_values[i], y: new_scatter_wapor_values[i] });
            }
        }
        if (valid_data.length > 1) {
            let sum_x = 0, sum_y = 0, sum_xy = 0, sum_x2 = 0, sum_y2 = 0;
            const n = valid_data.length;
            for (let i = 0; i < n; i++) {
                sum_x += valid_data[i].x; sum_y += valid_data[i].y;
                sum_xy += valid_data[i].x * valid_data[i].y;
                sum_x2 += valid_data[i].x * valid_data[i].x;
                sum_y2 += valid_data[i].y * valid_data[i].y;
            }
            const numerator = (n * sum_xy) - (sum_x * sum_y);
            const denominator_x = (n * sum_x2) - (sum_x * sum_x);
            const denominator_y = (n * sum_y2) - (sum_y * sum_y);
            if (denominator_x > 0 && denominator_y > 0) {
                const r = numerator / Math.sqrt(denominator_x * denominator_y);
                r2_val_scatter = (r * r).toFixed(3);
            }
        }
    }
    r2_label.text = `R² = ${r2_val_scatter}`;

    // Update axis ranges and 1:1 line
    let overall_min_val = data_type === 'Rainfall' ? global_min_rainfall : global_min_ret;
    let overall_max_val = data_type === 'Rainfall' ? global_max_rainfall : global_max_ret;

    p_scatter.x_range.start = overall_min_val;
    p_scatter.x_range.end = overall_max_val;
    p_scatter.y_range.start = overall_min_val;
    p_scatter.y_range.end = overall_max_val;

    if (p_scatter.renderers.length > 1 && p_scatter.renderers[1].data_source) {
        const one_to_one_line_source = p_scatter.renderers[1].data_source;
        one_to_one_line_source.data['x'] = [overall_min_val, overall_max_val];
        one_to_one_line_source.data['y'] = [overall_min_val, overall_max_val];
        one_to_one_line_source.change.emit();
    }

    scatter_source.change.emit();
    p_scatter.change.emit();
"""
# Link widgets to CustomJS callback
callback = CustomJS(args=dict(
    year_select=year_select,
    data_type_buttons=data_type_buttons,
    map_source_select=map_source_select,
    governorate_select=governorate_select,
    map_source_bokeh=initial_map_geojson_source, # This is the GeoJSONDataSource for the map
    color_mapper=color_mapper,
    p_map=p_map,
    p_scatter=p_scatter,
    scatter_source=scatter_source,
    r2_label=r2_label, # Pass the R-squared Label object directly
    all_dashboard_data_js=all_dashboard_data, # Pass the comprehensive data structure for scatterplot to JS
    all_map_data_for_js_callback=all_map_data_for_js, # Pass the pre-generated map GeoJSONs
    global_min_rainfall=global_min_rainfall,
    global_max_rainfall=global_max_rainfall,
    global_min_ret=global_min_ret,
    global_max_ret=global_max_ret,
    blues_palette=Blues256[::-1],
    rdbu_palette=RdBu11,
    greens_palette=Greens256[::-1],
    global_max_diff_rainfall=global_max_diff_rainfall,
    global_max_diff_ret=global_max_diff_ret
), code=callback_code)

year_select.js_on_change('value', callback)
data_type_buttons.js_on_change('active', callback)
map_source_select.js_on_change('value', callback)
governorate_select.js_on_change('value', callback)

# =============================================================================
# LAYOUT & DISPLAY
# =============================================================================
# Create layout
# Title for the dashboard
dashboard_title = Div(
    text="<h1>Iraq Climate Data Dashboard (WaPOR vs AgERA5)</h1>",
    width=1200, height=50
)

# Controls in a horizontal row
controls_row = bokeh_row(year_select, data_type_box, map_source_select, governorate_select) # Changed 'row' to 'bokeh_row'

# ✅ scatter wrapped with x label below and y label above
scatter_col = column(p_scatter)
# Plots in a horizontal row
plots_row = bokeh_row(p_map, scatter_col) # Changed 'row' to 'bokeh_row'
# Combined layout: Title, then controls row, then plots row
layout = column(dashboard_title, controls_row, plots_row)

show(layout)

## 🎯 Takeaways
Building interactive visualizations with Bokeh can greatly enhance how you explore and communicate your data. Now that you have a fully functional dashboard, here are some recommended next steps to keep learning:

* **🤖 Play with the Dashboard using Gemini:** Don't stop here! Try copying the dashboard code from the previous cell and asking **Gemini** to help you modify it. You can test prompts like:
  * *"How can I change the scatterplot points to be triangles instead of circles?"*
  * *"Can you help me add a new dropdown widget to change the color palette?"*
* **Explore Bokeh Documentation:** Bokeh is incredibly customizable. For example, you can learn how to format, style, and adjust the axes of your plots by visiting the [Bokeh Axes Documentation](https://docs.bokeh.org/en/latest/docs/user_guide/basic/axes.html).
*  **Bokeh in Jupyter Notebooks:** Interactive plots shine in notebook environments like Colab and Jupyter. Discover more advantages and advanced interactive features by checking out the [Bokeh Jupyter Notebooks User Guide](https://docs.bokeh.org/en/latest/docs/user_guide/output/jupyter.html) and their [Interactive Tutorial Gallery](https://mybinder.org/v2/gh/bokeh/bokeh-notebooks/master?filepath=tutorial%2F00%20-%20Introduction%20and%20Setup.ipynb).